# AMD ROCm Hardware Validation & End-to-End RAG Benchmark

**Project:** AMD AI League Mini-Challenge 3 (Retrieval-Augmented Generation)  
**Author:** Aryan Shete  
**Repository:** [github.com/Aryanshete/AMD-Retrieval-Augmented-Generation](https://github.com/Aryanshete/AMD-Retrieval-Augmented-Generation)  
**Target Architecture:** AMD Instinct MI210/MI250 / ROCm HIP (`rocm/pytorch`)  

---

### System Overview
This benchmark validates the end-to-end retrieval and question answering pipeline against the official evaluation constraints:
1. **Two-Phase CLI Contract:** Autonomous indexing pass (`--index /app/corpus`) followed by independent per-query executions (`--query-id <id> --query <text>`).
2. **Continuous VRAM Sampling:** Peak memory must reside strictly within **$1.0 \text{ GiB} \le \text{Peak VRAM} \le 48.0 \text{ GiB}$**.
3. **Zero Network Egress:** Offline-complete processing (`--network none`) with zero external dependency requests.
4. **Exact Citation Necessity:** Files are cited if and only if removing them renders answering impossible.

## 1. Accelerator & Runtime Diagnostics
Inspect the host PyTorch installation, device binding, and available compute accelerators.

In [ ]:
import os
import sys
import time
import torch

print("=" * 70)
print(" HARDWARE ACCELERATION ENVIRONMENT AUDIT")
print("=" * 70)
print(f"[*] Python Version      : {sys.version.split()[0]}")
print(f"[*] PyTorch Version     : {torch.__version__}")
print(f"[*] ROCm / HIP Compiled : {'rocm' in torch.__version__.lower()}")
print(f"[*] CUDA/HIP Available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_id = 0
    props = torch.cuda.get_device_properties(device_id)
    total_vram_gib = props.total_memory / (1024 ** 3)
    print(f"[*] Device [0] Name     : {props.name}")
    print(f"[*] Total Device Memory : {total_vram_gib:.2f} GiB")
    print(f"[*] Multi-Processors    : {props.multi_processor_count}")
else:
    print("[*] Warning: Operating in CPU development fallback mode.")
print("=" * 70)

## 2. Active Tensor Core & VRAM Compliance Verification
Executes native FP16 matrix operations on compute tensor cores, ensuring peak memory stays firmly within the required **1 – 48 GiB** evaluation boundary.

In [ ]:
from app.models.gpu_accelerator import ROCmGPUAccelerator

accelerator = ROCmGPUAccelerator()
print(f"Bound Device: {accelerator.device} ({accelerator.gpu_name})")

# Execute genuine tensor benchmark
bench_results = accelerator.run_gpu_benchmark(dim=1024, count=20000)

print("\n--- Benchmark Metrics ---")
for k, v in bench_results.items():
    print(f"  {k:<22}: {v}")

allocated_gib = accelerator.get_vram_usage_gib()
peak_gib = accelerator.get_peak_vram_usage_gib()
print(f"\nAllocated VRAM: {allocated_gib:.2f} GiB | Peak VRAM: {peak_gib:.2f} GiB")

if accelerator.device.startswith("cuda"):
    assert 1.0 <= peak_gib <= 48.0, f"VRAM peak {peak_gib:.2f} GiB is outside [1, 48] GiB!"
    print("[PASS] VRAM peak conforms to evaluation limits.")

## 3. Corpus Ingestion & Indexing Pass (`--index`)
Performs complete defensive traversal over all document modalities (PDF, DOCX, XLSX, CSV, PNG/JPG diagrams, and logs), generating persisted hybrid indices under `app/index`.

In [ ]:
from pathlib import Path
from app.indexing.crawler import crawl_corpus, persist_index
from app.retrieval.hybrid import HybridRetriever
from app.config import INDEX_DIR

corpus_path = Path("mc3-corpus").resolve()
print(f"Indexing corpus at: {corpus_path}")

t0 = time.time()
crawl_result = crawl_corpus(corpus_path)
persist_index(crawl_result, INDEX_DIR)

retriever = HybridRetriever()
retriever.build(crawl_result.all_chunks)
retriever.save(INDEX_DIR)
indexing_duration = time.time() - t0

print(f"[PASS] Ingestion complete in {indexing_duration:.2f}s")
print(f"  - Valid Documents : {len([d for d in crawl_result.documents if d.is_valid])}")
print(f"  - Total Chunks    : {len(crawl_result.all_chunks)}")
print(f"  - Isolated Errors : {len(crawl_result.failed_files)}")

## 4. End-to-End Regression Benchmark (Q1 – Q10)
Evaluates the 10 sample queries against the regression suite, measuring value accuracy, exact citation sets, and per-query latency.

In [ ]:
from app.eval.regression_suite import run_sample_regression

questions_file = Path("sample-questions.json").resolve()
results, all_passed = run_sample_regression(corpus_path, questions_file)

print("=" * 95)
print(f"{'ID':<5} | {'Answer':<16} | {'Status':<6} | {'Latency':<9} | {'Citations'}")
print("-" * 95)
for r in results:
    status_str = "PASS" if r.passed else "FAIL"
    cites_str = ", ".join(sorted(r.actual_citations)) if r.actual_citations else "[]"
    print(f"Q{r.n:<4} | {r.actual_answer:<16} | {status_str:<6} | {r.latency_ms:>6.1f} ms | {cites_str}")
print("=" * 95)

total_score = sum(20 for r in results if r.passed)
print(f"\nFinal Regression Score: {total_score} / 200 Points (100% Pass: {all_passed})")

## 5. Contract Compliance Certification
Verifies the Definition of Done exit criteria required for competition submission.

In [ ]:
import subprocess

# Run the official Definition of Done certification
proc = subprocess.run(
    [sys.executable, "-m", "unittest", "tests/test_definition_of_done.py"],
    capture_output=True,
    text=True,
)

if proc.returncode == 0:
    print("[CERTIFIED] All 8 Definition of Done criteria passed successfully.")
else:
    print("[FAIL] Certification failed:")
    print(proc.stderr)
print(proc.stdout)